# Business Entity Resolution: cross-encoder rescoring (GPU)

This notebook fine-tunes a multilingual transformer (`distilbert-base-multilingual-cased`, **Apache-2.0**, 134M parameters) to rescore the pairs where our LightGBM model was uncertain. It blends the two scores and writes a ready-to-upload `matching_results.tsv`.

**Before running**
1. Upload the `ber_colab` folder (from `Downloads/student_resource/ber_colab` on your Mac) to **My Drive**, so that the path is `My Drive/ber_colab/`.
2. Go to **Runtime → Change runtime type → T4 GPU**.
3. Click **Runtime → Run all**. It takes about 45–60 minutes. Progress is saved to Drive, so if Colab disconnects, just **Run all** again and it resumes.

Outputs land in `My Drive/ber_colab/out/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, time, math, numpy as np, pandas as pd, torch
hits = glob.glob('/content/drive/MyDrive/**/ce_train.parquet', recursive=True)
print('found:', hits)
assert hits, 'ber_colab not found in Drive yet - wait for the upload to finish, then re-run this cell'
D = os.path.dirname(hits[0])
OUT = f'{D}/out'; os.makedirs(OUT, exist_ok=True)
print('using', D, '->', sorted(os.listdir(D)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL = 'distilbert-base-multilingual-cased'   # Apache-2.0
MAXLEN = 96
tok = AutoTokenizer.from_pretrained(MODEL)

def side(name, addr):
    return (name.fillna('').astype(str) + ' | ' + addr.fillna('').astype(str)).tolist()

class PairData(torch.utils.data.Dataset):
    def __init__(self, a, b, y=None): self.a, self.b, self.y = a, b, y
    def __len__(self): return len(self.a)
    def __getitem__(self, i): return i

def collate_fn(ds):
    def f(idx):
        enc = tok([ds.a[i] for i in idx], [ds.b[i] for i in idx], truncation=True, max_length=MAXLEN,
                  padding=True, return_tensors='pt')
        if ds.y is not None: enc['labels'] = torch.tensor([ds.y[i] for i in idx], dtype=torch.float32)
        return enc
    return f

@torch.no_grad()
def predict(model, a, b, bs=512):
    model.eval(); ds = PairData(a, b)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, collate_fn=collate_fn(ds), num_workers=2)
    out = []
    for i, enc in enumerate(dl):
        enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(torch.sigmoid(model(**enc).logits.float().squeeze(-1)).cpu().numpy())
        if i % 200 == 0: print(f'  predict {i*bs}/{len(ds)}', flush=True)
    return np.concatenate(out)

In [ ]:
# ---- 1. fine-tune (skipped if a saved model exists)
CKPT = f'{D}/ce_model'
if os.path.exists(f'{CKPT}/config.json'):
    model = AutoModelForSequenceClassification.from_pretrained(CKPT).cuda()
    print('loaded saved model')
else:
    tr = pd.read_parquet(f'{D}/ce_train.parquet')
    a, b, y = side(tr.s1_name, tr.s1_addr), side(tr.r_name, tr.r_addr), tr.y.values.astype(np.float32)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()
    ds = PairData(a, b, y); BS = 128
    dl = torch.utils.data.DataLoader(ds, batch_size=BS, shuffle=True, collate_fn=collate_fn(ds), num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
    steps = len(dl); sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(); lossf = torch.nn.BCEWithLogitsLoss()
    model.train(); t0 = time.time(); run = 0
    for i, enc in enumerate(dl):
        enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}; lab = enc.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16):
            loss = lossf(model(**enc).logits.float().squeeze(-1), lab)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        run = 0.98 * run + 0.02 * loss.item()
        if i % 500 == 0: print(f'step {i}/{steps} loss {run:.4f} {time.time()-t0:.0f}s', flush=True)
    model.save_pretrained(CKPT); tok.save_pretrained(CKPT)
    del tr, a, b, y, ds, dl
print('model ready')

In [ ]:
# ---- 2. score the uncertain pairs (validation + test); cached on Drive
LO, HI = 0.02, 0.98
val = pd.read_parquet(f'{D}/val_pairs.parquet')
mv = ((val.p_lgb > LO) & (val.p_lgb < HI)).values
fv = f'{OUT}/val_ce.npy'
if not os.path.exists(fv):
    np.save(fv, predict(model, side(val.s1_name[mv], val.s1_addr[mv]), side(val.r_name[mv], val.r_addr[mv])))
val_ce = np.load(fv)
print('val uncertain pairs', mv.sum())

test = pd.read_parquet(f'{D}/test_pairs.parquet')
mt = ((test.p_lgb > LO) & (test.p_lgb < HI)).values
ft = f'{OUT}/test_ce.npy'
if not os.path.exists(ft):
    tt = pd.read_parquet(f'{D}/test_uncertain_text.parquet')   # rows = test[mt], same order
    assert len(tt) == mt.sum()
    np.save(ft, predict(model, side(tt.s1_name, tt.s1_addr), side(tt.r_name, tt.r_addr)))
    del tt
test_ce = np.load(ft)
print('test uncertain pairs', mt.sum())

In [ ]:
# ---- 3. blend on validation (logistic regression on the uncertain pairs) + threshold search by macro F0.5
from sklearn.linear_model import LogisticRegression
lg = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
def feats(pl, pc): return np.column_stack([lg(pl), lg(pc), lg(pl) * lg(pc)])
lr = LogisticRegression(C=1.0, max_iter=1000).fit(feats(val.p_lgb.values[mv], val_ce), val.y.values[mv])
print('blend coefs', lr.coef_, lr.intercept_)

def blended(df, m, ce):
    p = df.p_lgb.values.astype(np.float64).copy()
    p[m] = lr.predict_proba(feats(df.p_lgb.values[m], ce))[:, 1]
    return p

truth = pd.read_parquet(f'{D}/val_truth.parquet'); vids = pd.read_parquet(f'{D}/val_meta.parquet').val_s1.values
def decide(df, p, thr):
    d = pd.DataFrame({'s1': df.s1.values, 'r': df.r.values, 'p': p})
    d = d[d.p >= d.groupby('r').p.transform('max')]      # each record -> its best Source1 entity only
    return d[d.p >= thr][['s1', 'r']]
def f05(pred):
    u = pd.Index(vids)
    tp = pred.merge(truth, on=['s1', 'r']).groupby('s1').size().reindex(u).fillna(0).values
    npd = pred.groupby('s1').size().reindex(u).fillna(0).values
    nt = truth.groupby('s1').size().reindex(u).fillna(0).values
    f = np.where((npd == 0) & (nt == 0), 1.0, 0.0)
    m = tp > 0; P = tp[m] / npd[m]; R = tp[m] / nt[m]; f[m] = 1.25 * P * R / (0.25 * P + R)
    return f.mean()

pv_blend = blended(val, mv, val_ce)
res = []
for thr in [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    res.append((thr, f05(decide(val, val.p_lgb.values, thr)), f05(decide(val, pv_blend, thr))))
res = pd.DataFrame(res, columns=['thr', 'F05_lgb_only', 'F05_blend']); print(res.to_string(index=False))
best = res.loc[res.F05_blend.idxmax()]; print('best blend thr', best.thr, 'F0.5', round(best.F05_blend, 5),
      '| lgb-only best', round(res.F05_lgb_only.max(), 5))

In [ ]:
# ---- 4. write submission files (every Source1 entity gets a row)
ids = pd.read_parquet(f'{D}/test_ids.parquet').entity_id.values
rids = pd.read_parquet(f'{D}/test_r_ids.parquet').entity_id.values
pt = blended(test, mt, test_ce)
def write(pairs, path, col):
    pairs = pairs.sort_values(['s1', 'r'])
    s1v = pairs.s1.values; rs = rids[pairs.r.values]
    starts = np.flatnonzero(np.r_[True, s1v[1:] != s1v[:-1]])
    joined = [','.join(x) for x in np.split(rs, starts[1:])] if len(rs) else []
    c = np.full(len(ids), '', dtype=object); c[s1v[starts]] = joined
    pd.DataFrame({'source1_entity_id': ids, col: c}).to_csv(path, sep='\t', index=False)
    print('wrote', path)
thr = float(best.thr)
write(decide(test, pt, thr), f'{OUT}/matching_results.tsv', 'matched_entity_ids')
write(decide(test, pt, min(thr + 0.1, 0.95)), f'{OUT}/matching_results_stricter.tsv', 'matched_entity_ids')
write(test[['s1', 'r']], f'{OUT}/candidate_pairs.tsv', 'candidate_entity_ids')
print('matches per entity:', len(decide(test, pt, thr)) / len(ids))
print('DONE - download My Drive/ber_colab/out/matching_results.tsv and upload it to the leaderboard')

In [ ]:
# ---- 5. second, stronger model: XLM-RoBERTa-base (MIT licence). Cached on Drive.
MODEL2 = 'xlm-roberta-base'
CK2 = f'{D}/ce_model_xlmr'
tok = AutoTokenizer.from_pretrained(MODEL2)          # collate_fn/predict use this global tokenizer
if os.path.exists(f'{CK2}/config.json'):
    model2 = AutoModelForSequenceClassification.from_pretrained(CK2).cuda(); print('loaded saved xlmr')
else:
    tr = pd.read_parquet(f'{D}/ce_train.parquet')
    a, b, y = side(tr.s1_name, tr.s1_addr), side(tr.r_name, tr.r_addr), tr.y.values.astype(np.float32)
    model2 = AutoModelForSequenceClassification.from_pretrained(MODEL2, num_labels=1).cuda()
    ds = PairData(a, b, y); BS = 64
    dl = torch.utils.data.DataLoader(ds, batch_size=BS, shuffle=True, collate_fn=collate_fn(ds), num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model2.parameters(), lr=2e-5, weight_decay=0.01)
    steps = len(dl); sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler('cuda'); lossf = torch.nn.BCEWithLogitsLoss()
    model2.train(); t0 = time.time(); run = 0
    for i, enc in enumerate(dl):
        enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}; lab = enc.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16):
            loss = lossf(model2(**enc).logits.float().squeeze(-1), lab)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        run = 0.98 * run + 0.02 * loss.item()
        if i % 1000 == 0: print(f'step {i}/{steps} loss {run:.4f} {time.time()-t0:.0f}s', flush=True)
    model2.save_pretrained(CK2); tok.save_pretrained(CK2)
    del tr, a, b, y, ds, dl
fv2, ft2 = f'{OUT}/val_ce2.npy', f'{OUT}/test_ce2.npy'
if not os.path.exists(fv2):
    np.save(fv2, predict(model2, side(val.s1_name[mv], val.s1_addr[mv]), side(val.r_name[mv], val.r_addr[mv])))
if not os.path.exists(ft2):
    tt = pd.read_parquet(f'{D}/test_uncertain_text.parquet')
    np.save(ft2, predict(model2, side(tt.s1_name, tt.s1_addr), side(tt.r_name, tt.r_addr))); del tt
val_ce2, test_ce2 = np.load(fv2), np.load(ft2)
print('xlmr scored', len(val_ce2), len(test_ce2))

In [ ]:
# ---- 6. three-way blend (LightGBM + distilbert + XLM-R), compare, write v2 files
def feats3(pl, c1, c2):
    return np.column_stack([lg(pl), lg(c1), lg(c2), lg(pl) * lg(c1), lg(pl) * lg(c2)])
lr3 = LogisticRegression(C=1.0, max_iter=2000).fit(feats3(val.p_lgb.values[mv], val_ce, val_ce2), val.y.values[mv])
print('3-way coefs', lr3.coef_.round(3), lr3.intercept_.round(3))
def blended3(df, m, c1, c2):
    p = df.p_lgb.values.astype(np.float64).copy()
    p[m] = lr3.predict_proba(feats3(df.p_lgb.values[m], c1, c2))[:, 1]
    return p
pv3 = blended3(val, mv, val_ce, val_ce2)
rows = [(t, f05(decide(val, pv_blend, t)), f05(decide(val, pv3, t))) for t in [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]]
res3 = pd.DataFrame(rows, columns=['thr', 'F05_blend_v1', 'F05_blend_v2_3way']); print(res3.to_string(index=False))
b3 = res3.loc[res3.F05_blend_v2_3way.idxmax()]; print('best 3-way thr', b3.thr, 'F0.5', round(b3.F05_blend_v2_3way, 5))
pt3 = blended3(test, mt, test_ce, test_ce2)
write(decide(test, pt3, float(b3.thr)), f'{OUT}/matching_results_v2.tsv', 'matched_entity_ids')
write(decide(test, pt3, min(float(b3.thr) + 0.1, 0.95)), f'{OUT}/matching_results_v2_stricter.tsv', 'matched_entity_ids')
print('DONE: new files matching_results_v2.tsv / _v2_stricter.tsv in My Drive/ber_colab/out/')

In [ ]:
# ---- 7. SIBLING-AWARE model: sees  S1  vs  candidate ## best already-matched record of the same S1
MODEL3 = 'distilbert-base-multilingual-cased'   # Apache-2.0
CK3 = f'{D}/ce_model_sib'
tok = AutoTokenizer.from_pretrained(MODEL3); MAXLEN = 128
def s_(x): return pd.Series(np.asarray(x, dtype=object)).fillna('').astype(str)
def side_sib(n, a, sn, sa): return (s_(n) + ' | ' + s_(a) + ' ## ' + s_(sn) + ' | ' + s_(sa)).tolist()
if os.path.exists(f'{CK3}/config.json'):
    model3 = AutoModelForSequenceClassification.from_pretrained(CK3).cuda(); print('loaded saved sibling model')
else:
    tr = pd.read_parquet(f'{D}/ce_train_sib.parquet')
    a = side(tr.s1_name, tr.s1_addr)
    b = side_sib(tr.r_name.values, tr.r_addr.values, tr.sib_name.values, tr.sib_addr.values)
    y = tr.y.values.astype(np.float32); del tr
    model3 = AutoModelForSequenceClassification.from_pretrained(MODEL3, num_labels=1).cuda()
    ds = PairData(a, b, y); BS = 128
    dl = torch.utils.data.DataLoader(ds, batch_size=BS, shuffle=True, collate_fn=collate_fn(ds), num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model3.parameters(), lr=3e-5, weight_decay=0.01)
    steps = len(dl); sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.amp.GradScaler('cuda'); lossf = torch.nn.BCEWithLogitsLoss()
    model3.train(); t0 = time.time(); run = 0
    for i, enc in enumerate(dl):
        enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}; lab = enc.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16):
            loss = lossf(model3(**enc).logits.float().squeeze(-1), lab)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        run = 0.98 * run + 0.02 * loss.item()
        if i % 500 == 0: print(f'step {i}/{steps} loss {run:.4f} {time.time()-t0:.0f}s', flush=True)
    model3.save_pretrained(CK3); tok.save_pretrained(CK3)
    del a, b, y, ds, dl
fv3, ft3 = f'{OUT}/val_ce_sib.npy', f'{OUT}/test_ce_sib.npy'
if not os.path.exists(fv3):
    vs = pd.read_parquet(f'{D}/val_sib_text.parquet'); assert len(vs) == mv.sum()      # rows = val[mv]
    np.save(fv3, predict(model3, side(val.s1_name[mv], val.s1_addr[mv]),
                         side_sib(val.r_name.values[mv], val.r_addr.values[mv], vs.sib_name.values, vs.sib_addr.values)))
if not os.path.exists(ft3):
    tt = pd.read_parquet(f'{D}/test_uncertain_text.parquet'); ts = pd.read_parquet(f'{D}/test_sib_text.parquet')
    assert len(tt) == len(ts) == mt.sum()                                               # rows = test[mt]
    np.save(ft3, predict(model3, side(tt.s1_name, tt.s1_addr),
                         side_sib(tt.r_name.values, tt.r_addr.values, ts.sib_name.values, ts.sib_addr.values)))
    del tt, ts
val_ce3, test_ce3 = np.load(fv3), np.load(ft3)
print('sibling model scored', len(val_ce3), len(test_ce3))

In [ ]:
# ---- 8. blend everything available (LightGBM + distilbert + sibling model [+ XLM-R if run]), compare, write v3
cols_v = [val_ce, val_ce3]; cols_t = [test_ce, test_ce3]; names = ['distilbert', 'sibling']
if os.path.exists(f'{OUT}/val_ce2.npy') and os.path.exists(f'{OUT}/test_ce2.npy'):
    cols_v.append(np.load(f'{OUT}/val_ce2.npy')); cols_t.append(np.load(f'{OUT}/test_ce2.npy')); names.append('xlmr')
def featsN(pl, cs): L = lg(pl); return np.column_stack([L] + [lg(c) for c in cs] + [L * lg(c) for c in cs])
lrN = LogisticRegression(C=1.0, max_iter=3000).fit(featsN(val.p_lgb.values[mv], cols_v), val.y.values[mv])
print('models blended:', names, '| coefs', lrN.coef_.round(3))
def blendedN(df, m, cs):
    p = df.p_lgb.values.astype(np.float64).copy(); p[m] = lrN.predict_proba(featsN(df.p_lgb.values[m], cs))[:, 1]; return p
pvN = blendedN(val, mv, cols_v)
rows = [(t, f05(decide(val, pv_blend, t)), f05(decide(val, pvN, t))) for t in [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]]
resN = pd.DataFrame(rows, columns=['thr', 'F05_blend_v1', 'F05_blend_v3']); print(resN.to_string(index=False))
bN = resN.loc[resN.F05_blend_v3.idxmax()]; print('best v3 thr', bN.thr, 'F0.5', round(bN.F05_blend_v3, 5), '| v1 best', round(resN.F05_blend_v1.max(), 5))
ptN = blendedN(test, mt, cols_t)
write(decide(test, ptN, float(bN.thr)), f'{OUT}/matching_results_v3.tsv', 'matched_entity_ids')
write(decide(test, ptN, min(float(bN.thr) + 0.1, 0.95)), f'{OUT}/matching_results_v3_stricter.tsv', 'matched_entity_ids')
print('DONE: matching_results_v3.tsv and matching_results_v3_stricter.tsv in My Drive/ber_colab/out/')

In [ ]:
# ---- 9. final files: A = stricter everywhere, B = stricter + France extra-strict
P_BEST = ptN if 'ptN' in globals() else (pt3 if 'pt3' in globals() else pt)
print('using', 'v3' if 'ptN' in globals() else ('v2' if 'pt3' in globals() else 'v1'))
ctry = pd.read_parquet(f'{D}/test_country.parquet').country.values
isfr = ctry[test.s1.values] == 'France'
def decide_c(df, p, thr_row):
    d = pd.DataFrame({'s1': df.s1.values, 'r': df.r.values, 'p': p, 't': thr_row})
    d = d[d.p >= d.groupby('r').p.transform('max')]          # each record -> its best Source1 only
    return d[d.p >= d.t][['s1', 'r']]
write(decide_c(test, P_BEST, np.full(len(test), 0.85)), f'{OUT}/final_A_strict.tsv', 'matched_entity_ids')
write(decide_c(test, P_BEST, np.where(isfr, 0.92, 0.85)), f'{OUT}/final_B_strict_franceX.tsv', 'matched_entity_ids')
print('DONE: final_A_strict.tsv and final_B_strict_franceX.tsv in My Drive/ber_colab/out/')

In [ ]:
for t in (0.88, 0.90, 0.92):
    write(decide_c(test, P_BEST, np.full(len(test), t)), f'{OUT}/final_thr{int(t*100)}.tsv', 'matched_entity_ids')
    print(t, 'done')